# Parkinson's Disease Detection from Voice Measurements

## Objective
This beginner-friendly project explores whether numerical voice measurements in a public dataset can be used to classify its recorded `status` label. We compare Logistic Regression and Random Forest using a held-out test split. A model's output is a statistical prediction, not a diagnosis.

## Dataset and source
We use the genuine **Parkinsons** dataset from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/174/parkinsons). The official [ZIP download](https://archive.ics.uci.edu/static/public/174/parkinsons.zip) contains `parkinsons.data`: 195 voice recordings, 22 numeric acoustic features, the recording identifier `name`, and binary target `status` (1 = Parkinson's case, 0 = control). `name` identifies a recording and must not be used as a predictor.

Download that ZIP, extract `parkinsons.data` into this project's `data` folder, and confirm the path is `data/parkinsons.data`. The download steps are also in `data/README.md`. This project does not include or fabricate patient records. If the file is absent, the notebook explains how to obtain it and skips data-dependent analysis, training, evaluation, and saving; an export in that state is not a completed analysis.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "train_model.py").is_file():
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from train_model import (
    build_models,
    evaluate_model,
    load_dataset,
    save_models,
    split_features_target,
)

DATA_PATH = PROJECT_DIR / "data" / "parkinsons.data"
print(f"Project folder: {PROJECT_DIR}")
print(f"Expected dataset: {DATA_PATH}")
try:
    df = load_dataset(DATA_PATH)
except FileNotFoundError as error:
    df = None
    print(error)
    print("Next step: download the official UCI ZIP, extract parkinsons.data into the data folder, then rerun this notebook.")

## Inspect the data

We first inspect dimensions, column types, and sample rows. If the real CSV is missing, these cells intentionally skip instead of manufacturing data.

In [ ]:
if df is not None:
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    display(df.head())
    display(df.info())
else:
    print("Skipped: real dataset is not available.")

## Missing values, duplicate rows, and class balance

Missing values in input features can be handled by the median imputer inside each training pipeline. We report duplicate counts but do not silently remove rows. The class counts help us understand imbalance. The UCI dataset contains more records in one class; the actual counts below are calculated from the downloaded file.

In [ ]:
if df is not None:
    print("Missing values by column:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
    duplicate_count = int(df.duplicated().sum())
    print(f"Exact duplicate rows (reported, not removed): {duplicate_count}")
    print("Class counts (0 = control, 1 = Parkinson's case):")
    display(df["status"].value_counts(dropna=False).sort_index().to_frame("count"))
else:
    print("Skipped: real dataset is not available.")

## Exploratory data analysis

The plots below show target counts, the distribution of one example voice feature by target class, and a correlation overview for numeric columns. These are descriptive views of the downloaded dataset?not diagnostic thresholds or proof of causation.

In [ ]:
if df is not None:
    numeric_features = df.drop(columns=["status", "name"], errors="ignore").apply(pd.to_numeric, errors="raise")
    fig, axes = plt.subplots(1, 3, figsize=(19, 5))

    df["status"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["steelblue", "darkorange"])
    axes[0].set(title="Recorded class counts", xlabel="status (0=control, 1=case)", ylabel="recordings")
    axes[0].tick_params(axis="x", rotation=0)

    example_feature = numeric_features.columns[0]
    for label, group in df.groupby("status"):
        axes[1].hist(pd.to_numeric(group[example_feature]), bins=15, alpha=0.55, label=f"status {label}")
    axes[1].set(title=f"{example_feature} distribution", xlabel=example_feature, ylabel="recordings")
    axes[1].legend()

    correlation = numeric_features.corr()
    image = axes[2].imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    axes[2].set(title="Voice-feature correlations", xticks=[], yticks=[])
    fig.colorbar(image, ax=axes[2], fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()
else:
    print("Skipped: real dataset is not available.")

## Separate features and target

The target is `status`. The recording identifier `name` is excluded because it is not a voice measurement and could let a model memorize file naming patterns. All remaining columns are converted to numeric values.

In [ ]:
if df is not None:
    X, y = split_features_target(df)
    print(f"Feature matrix shape: {X.shape}")
    print(f"Target shape: {y.shape}; classes: {sorted(y.unique())}")
    print("Feature columns:")
    display(pd.DataFrame({"feature": X.columns}))
else:
    X, y = None, None
    print("Skipped: real dataset is not available.")

## Train/test split and preprocessing

We reserve 20% of the records for a final holdout test and stratify the split so both classes are represented in each part when possible. Preprocessing is inside each scikit-learn `Pipeline`: missing numeric values are imputed using the training data only, and Logistic Regression also scales features using training data only. This prevents preprocessing from learning from the held-out test rows. Random Forest does not need feature scaling, so its pipeline uses imputation only.

In [ ]:
if X is not None:
    from sklearn.model_selection import train_test_split

    if y.value_counts().min() < 2:
        raise ValueError("At least two real records per class are required for a stratified train/test split.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Training rows: {len(X_train)} | Holdout rows: {len(X_test)}")
    display(pd.DataFrame({"train": y_train.value_counts().sort_index(), "test": y_test.value_counts().sort_index()}))
else:
    X_train = X_test = y_train = y_test = None
    print("Skipped: real dataset is not available.")

## Fit Logistic Regression and Random Forest

Both algorithms use fixed random seeds for reproducibility. Balanced class weights are used to reduce the tendency to favor the more frequent class. They do not eliminate small-sample uncertainty or make predictions clinically reliable.

In [ ]:
if X_train is not None:
    models = build_models()
    for model in models.values():
        model.fit(X_train, y_train)
    print("Both pipelines were fitted on the training split.")
else:
    models = None
    print("Skipped: real dataset is not available; no model was trained.")

## Evaluate on the held-out test split

- **Confusion matrix:** counts of true versus predicted classes; rows are true labels (0, 1) and columns are predicted labels (0, 1).
- **Precision:** among records predicted as class 1, the fraction that are class 1.
- **Recall:** among actual class 1 records, the fraction detected as class 1.
- **F1-score:** the harmonic mean of precision and recall.
- **Classification report:** per-class precision, recall, F1, and support (number of true records).
- **ROC-AUC:** measures ranking across thresholds; calculated only if both classes are present in the held-out labels. These measures describe this small dataset split only.

In [ ]:
if models is not None:
    from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

    evaluations = {}
    for name, model in models.items():
        result = evaluate_model(model, X_test, y_test)
        evaluations[name] = result
        print(f"\n{name}")
        print(f"Confusion matrix (true rows, predicted columns; labels 0, 1):\n{result['confusion_matrix']}")
        print(f"Precision: {result['precision']:.3f}")
        print(f"Recall: {result['recall']:.3f}")
        print(f"F1-score: {result['f1']:.3f}")
        print(result["classification_report"])
        if "roc_auc" in result:
            print(f"ROC-AUC: {result['roc_auc']:.3f}")
        else:
            print("ROC-AUC omitted: the holdout has only one class.")

        ConfusionMatrixDisplay(
            confusion_matrix=result["confusion_matrix"], display_labels=["0 control", "1 case"]
        ).plot(cmap="Blues")
        plt.title(f"{name}: holdout confusion matrix")
        plt.tight_layout()
        plt.show()

    if y_test.nunique() == 2:
        fig, ax = plt.subplots(figsize=(7, 6))
        for name, model in models.items():
            RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax.set_title("Holdout ROC curves (descriptive only)")
        plt.tight_layout()
        plt.show()

    metric_table = pd.DataFrame(
        {name: {key: result[key] for key in ("precision", "recall", "f1", "accuracy", "roc_auc") if key in result}
         for name, result in evaluations.items()}
    ).T
    display(metric_table)
else:
    evaluations = None
    print("Skipped: real dataset is not available; no evaluation results exist.")

## Save fitted pipelines

When real data is present and training succeeds, save both fitted pipelines and their expected feature order with `joblib`. The Streamlit demo reads this artifact. No model artifact is created by the missing-data path.

In [ ]:
if models is not None:
    artifact_path = PROJECT_DIR / "submission" / "parkinsons_models.joblib"
    save_models(models, list(X.columns), artifact_path)
    print(f"Saved fitted pipelines: {artifact_path}")
else:
    print("Skipped: no real-data model was trained, so nothing was saved.")

## Conclusion and limitations

The cells above train and evaluate only when the official UCI CSV is present. Any displayed scores come from that downloaded file and this fixed holdout split; they are not clinical performance claims. The dataset is small, measurements may be correlated, and class imbalance can make accuracy misleading. A single split has substantial uncertainty and does not establish generalization to new populations, recording equipment, or clinical settings. The project is educational and must never be used to diagnose, screen, treat, or make medical decisions.

If the dataset was missing, the analysis is intentionally incomplete: follow the download instructions at the top of this notebook and rerun it before exporting a completed report.